# Assignment — Collaborative Filtering & Neural Network


# Question 1: Collaborative Filtering

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine

df = pd.read_csv('/Users/yuangzou/Downloads/radio_songs.csv')
print('Shape:', df.shape)
print(df.head(3))

In [ ]:
# Data is already a user-item matrix
# First column = user, remaining columns = artist names with 0/1 values
user_item = df.set_index('user')
print('User-item matrix shape:', user_item.shape)
print('Users:', user_item.shape[0], '  Songs:', user_item.shape[1])
user_item.head(3)

## Part A — Item-Item Collaborative Filtering (u2 & pink floyd)

Recommend 10 songs to users who listened to 'u2' and 'pink floyd'.
`similarity = 1 - cosine(item1, item2)`

In [ ]:
# item_matrix: rows=songs, cols=users
item_matrix = user_item.T
all_songs = item_matrix.index.tolist()

# Find u2 and pink floyd in the song list
target_songs = [s for s in all_songs if 'u2' in s.lower() or 'pink floyd' in s.lower()]
print('Target songs found:', target_songs)

# Compute average cosine similarity of every other song to the target songs
similarity_scores = {}
for song in all_songs:
    if song in target_songs:
        continue
    sims = []
    for target in target_songs:
        sim = 1 - cosine(item_matrix.loc[target].values, item_matrix.loc[song].values)
        sims.append(sim)
    similarity_scores[song] = sum(sims) / len(sims)

sim_series = pd.Series(similarity_scores).sort_values(ascending=False)
top10 = sim_series.head(10)

print('\n=== Top 10 Recommendations for listeners of u2 & pink floyd ===')
print('(Item-Item Collaborative Filtering — Cosine Similarity)\n')
for rank, (song, score) in enumerate(top10.items(), 1):
    print(f'{rank:2d}. {song:<40s}  similarity = {score:.4f}')

## Part B — User-User Collaborative Filtering for User 1606

Find the most similar user to 1606 using cosine similarity.
Recommend songs from the most similar user that 1606 has not heard.

In [ ]:
TARGET_USER = 1606
target_vec = user_item.loc[TARGET_USER].values

user_similarities = {}
for user in user_item.index:
    if user == TARGET_USER:
        continue
    sim = 1 - cosine(target_vec, user_item.loc[user].values)
    user_similarities[user] = sim

user_sim_series = pd.Series(user_similarities).sort_values(ascending=False)
most_similar_user  = user_sim_series.idxmax()
most_similar_score = user_sim_series.max()

print(f'Most similar user to user {TARGET_USER}: User {most_similar_user}')
print(f'Cosine similarity score: {most_similar_score:.4f}')
print('\nTop 5 most similar users:')
print(user_sim_series.head())

In [ ]:
songs_1606    = set(user_item.columns[user_item.loc[TARGET_USER] == 1])
songs_similar = set(user_item.columns[user_item.loc[most_similar_user] == 1])

recommendations_B = songs_similar - songs_1606

print(f'Songs user {TARGET_USER} has listened to ({len(songs_1606)} total):')
print(sorted(songs_1606))

print(f'\nSongs user {most_similar_user} has listened to ({len(songs_similar)} total):')
print(sorted(songs_similar))

print(f'\n=== Part B: Recommended songs for user {TARGET_USER} ===')
print(f'(Songs from most similar user {most_similar_user} not yet heard by {TARGET_USER})\n')
for i, song in enumerate(sorted(recommendations_B), 1):
    print(f'{i:2d}. {song}')

## Part C — How many recommended songs has user 1606 already heard?

In [ ]:
already_heard = songs_similar & songs_1606
not_yet_heard  = songs_similar - songs_1606

print(f'Total songs from most similar user (user {most_similar_user}): {len(songs_similar)}')
print(f'Already heard by user {TARGET_USER}: {len(already_heard)}')
print(f'Not yet heard (new recommendations): {len(not_yet_heard)}')
print()
print(f'=== Part C Answer: {len(already_heard)} song(s) already heard by user {TARGET_USER} ===')
print()
if already_heard:
    print('Songs already heard:')
    for s in sorted(already_heard):
        print(f'  - {s}')
else:
    print('None — user 1606 has not heard any of the songs from the most similar user.')

## Part D — User-Item Hybrid Recommendation Score

For each song user 1606 listened to:
1. Get top 10 similar songs + their similarity score
2. For each of those songs, get the user's purchase history (1 or 0)
3. Score = Σ(purchase × similarity) / Σ(similarity)
4. Return top 5 recommendations for user 1606

In [ ]:
# Build item-item similarity matrix (vectorized)
print('Building item-item similarity matrix...')
item_vecs = item_matrix.values.astype(float)
norms = np.linalg.norm(item_vecs, axis=1, keepdims=True)
norms[norms == 0] = 1
item_vecs_norm = item_vecs / norms
sim_matrix = item_vecs_norm @ item_vecs_norm.T
sim_df = pd.DataFrame(sim_matrix, index=all_songs, columns=all_songs)
print(f'Done. Shape: {sim_df.shape}')

user_row = user_item.loc[TARGET_USER]
listened_songs = user_row[user_row == 1].index.tolist()
print(f'\nUser {TARGET_USER} listened to {len(listened_songs)} songs: {listened_songs}')

In [ ]:
# Compute recommendation score for every candidate song
rec_scores = {}

for candidate in user_item.columns:
    numerator   = 0.0
    denominator = 0.0
    for listened in listened_songs:
        top10_sim = sim_df.loc[listened].drop(listened).nlargest(10)
        if candidate in top10_sim.index:
            sim_score    = top10_sim[candidate]
            purchase     = user_row[listened]   # 1 = listened
            numerator   += purchase * sim_score
            denominator += sim_score
    if denominator > 0:
        rec_scores[candidate] = numerator / denominator

rec_series = pd.Series(rec_scores).sort_values(ascending=False)
# Remove songs already listened to
rec_new = rec_series.drop(labels=[s for s in listened_songs if s in rec_series.index])

print(f'=== Part D: Top 5 Recommendations for User {TARGET_USER} (User-Item Hybrid) ===\n')
for rank, (song, score) in enumerate(rec_new.head(5).items(), 1):
    print(f'{rank}. {song:<40s}  score = {score:.4f}')

---
# Question 2: Neural Network using NumPy

Recalculate delta change in weight b2 for learning rates **0.0001, 0.01, 1** over **10 iterations**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Standard XOR training data (same as example NN notebook)
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([[0],[1],[1],[0]], dtype=float)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

def run_nn(learning_rate, n_iterations=10, seed=42):
    np.random.seed(seed)
    W1 = np.random.randn(2, 4)
    b1 = np.zeros((1, 4))
    W2 = np.random.randn(4, 1)
    b2 = np.zeros((1, 1))

    delta_b2_history = []
    loss_history     = []

    for i in range(n_iterations):
        # Forward pass
        Z1 = X @ W1 + b1
        A1 = sigmoid(Z1)
        Z2 = A1 @ W2 + b2
        A2 = sigmoid(Z2)

        loss = np.mean((y - A2) ** 2)
        loss_history.append(loss)

        # Backward pass
        dL_dA2  = -2 * (y - A2) / len(y)
        dA2_dZ2 = sigmoid_deriv(Z2)
        delta2  = dL_dA2 * dA2_dZ2

        dL_dW2 = A1.T @ delta2
        dL_db2 = np.sum(delta2, axis=0, keepdims=True)

        delta1  = (delta2 @ W2.T) * sigmoid_deriv(Z1)
        dL_dW1  = X.T @ delta1
        dL_db1  = np.sum(delta1, axis=0, keepdims=True)

        # Delta b2 = learning_rate * gradient
        delta_b2 = learning_rate * dL_db2
        delta_b2_history.append(float(delta_b2))

        # Update parameters
        W2 -= learning_rate * dL_dW2
        b2 -= delta_b2
        W1 -= learning_rate * dL_dW1
        b1 -= learning_rate * dL_db1

    return delta_b2_history, loss_history

learning_rates = [0.0001, 0.01, 1]
results = {}
for lr in learning_rates:
    delta_b2, losses = run_nn(learning_rate=lr, n_iterations=10)
    results[lr] = {'delta_b2': delta_b2, 'losses': losses}

# Print results table
print('=== Delta Change in Weight b2 across 10 Iterations ===\n')
print(f'{"Iter":<6}{"lr=0.0001":<22}{"lr=0.01":<22}{"lr=1"}')
print('-' * 72)
for i in range(10):
    row = f'{i+1:<6}'
    for lr in learning_rates:
        row += f'{results[lr]["delta_b2"][i]:<22.8f}'
    print(row)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'seagreen', 'tomato']
iters  = range(1, 11)

for lr, color in zip(learning_rates, colors):
    axes[0].plot(iters, results[lr]['delta_b2'], marker='o', label=f'lr={lr}', color=color, linewidth=2)
    axes[1].plot(iters, results[lr]['losses'],   marker='o', label=f'lr={lr}', color=color, linewidth=2)

axes[0].set_title('Delta Change in Weight b2 per Iteration', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Δb2')
axes[0].legend(); axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[0].grid(alpha=0.3)

axes[1].set_title('Training Loss (MSE) per Iteration', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('MSE Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/yuangzou/Downloads/nn_delta_b2.png', dpi=150)
plt.show()
print('Plot saved.')

In [ ]:
print('=== CONCLUSION: Effect of Learning Rate on Delta b2 ===\n')

for lr in learning_rates:
    db2 = results[lr]['delta_b2']
    print(f'Learning rate = {lr}:')
    print(f'  Iter 1  delta_b2 = {db2[0]:.8f}')
    print(f'  Iter 10 delta_b2 = {db2[-1]:.8f}')
    print()

print('---------------------------------------------------------------')
print('ANALYSIS:\n')
print('1. lr = 0.0001 (very small):')
print('   Delta b2 is extremely small at every iteration.')
print('   Weight barely updates → very slow convergence.')
print('   Loss decreases very little over 10 iterations.')
print()
print('2. lr = 0.01 (moderate):')
print('   Delta b2 is 100x larger than lr=0.0001 but remains stable.')
print('   Consistent gradient steps toward the minimum.')
print('   Loss decreases smoothly — best convergence behavior.')
print()
print('3. lr = 1 (large):')
print('   Delta b2 is very large initially, may oscillate.')
print('   Large updates risk overshooting the minimum.')
print('   Loss may not decrease monotonically — risk of divergence.')
print()
print('KEY TAKEAWAY:')
print('  delta_b2 = learning_rate × gradient')
print('  Learning rate directly scales the magnitude of every weight update.')
print('  Too small → painfully slow learning.')
print('  Too large → unstable or divergent training.')
print('  A moderate learning rate (e.g. 0.01) gives stable, efficient convergence.')